# 02 — Retrieval Pipeline Explorer

Compare all four retrieval stages side-by-side:

| Stage | Module | What it does |
|-------|--------|--------------|
| **Semantic** | `VectorSearcher` | Cosine similarity via embeddings |
| **BM25** | `KeywordSearcher` | Full-text / term frequency |
| **Hybrid** | `reciprocal_rank_fusion` | Rank-level merge of the two |
| **Reranked** | `CrossEncoderReranker` | Cross-encoder rescoring |

The final comparison table highlights the **reranked winners in green** across all four columns
so you can see where each final chunk originally ranked.

## 0 — Parameters (tweak these!)

In [2]:
# ── Query ─────────────────────────────────────────────
QUERY_TEXT = "French Revolution"

# ── Retrieval knobs ──────────────────────────────────
KEYWORD_TOP_K  = 20     # BM25 candidates
VECTOR_TOP_K   = 20     # semantic candidates
RRF_K          = 60     # RRF constant (higher = less rank-sensitive)
MERGE_TOP_N    = 30     # candidates kept after RRF merge
RERANK_TOP_N   = 5      # final winners after cross-encoder

# ── Display ──────────────────────────────────────────
SNIPPET_CHARS  = 80     # how many chars of chunk text to show per cell

# ── Paths ────────────────────────────────────────────
DB_PATH        = "../data/indexes/active/lancedb"
EMBED_MODEL    = "intfloat/multilingual-e5-base"
RERANK_MODEL   = "BAAI/bge-reranker-v2-m3"
DEVICE         = "mps"

## 1 — Initialise components

In [3]:
from pathlib import Path

from workbench.core.types import Query
from workbench.data_build.embeddings import SentenceTransformerEmbedder
from workbench.retrieval.keyword_search import LanceDBKeywordSearcher
from workbench.retrieval.vector_search import LanceDBVectorSearcher
from workbench.retrieval.hybrid import reciprocal_rank_fusion
from workbench.retrieval.rerankers import CrossEncoderReranker

db_path = Path(DB_PATH)

embedder = SentenceTransformerEmbedder(
    model_name=EMBED_MODEL, device=DEVICE,
)
kw_searcher  = LanceDBKeywordSearcher(db_path=db_path)
vec_searcher = LanceDBVectorSearcher(db_path=db_path, embedder=embedder)
reranker     = CrossEncoderReranker(model_name=RERANK_MODEL)

print(f"Embedder:  {embedder}")
print(f"Keyword:   {kw_searcher}")
print(f"Vector:    {vec_searcher}")
print(f"Reranker:  {reranker}")

/Users/alistair/Documents/101_Coding/Projects/AgenticAI/agentic_workbench/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2638.70it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2191.19it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]              


Embedder:  SentenceTransformerEmbedder(model='intfloat/multilingual-e5-base', dim=768, device='mps:0', query_prefix='query: ', passage_prefix='passage: ')
Keyword:   LanceDBKeywordSearcher(db=../data/indexes/active/lancedb, table='chunks')
Vector:    LanceDBVectorSearcher(db=../data/indexes/active/lancedb, table='chunks')
Reranker:  CrossEncoderReranker(model='BAAI/bge-reranker-v2-m3')


## 2 — Run all four stages

In [4]:
import time

query = Query(text=QUERY_TEXT)

# Stage 1: Keyword (BM25)
t0 = time.time()
kw_results = kw_searcher.search(query, top_k=KEYWORD_TOP_K)
kw_ms = (time.time() - t0) * 1000

# Stage 2: Vector (semantic)
t0 = time.time()
vec_results = vec_searcher.search(query, top_k=VECTOR_TOP_K)
vec_ms = (time.time() - t0) * 1000

# Stage 3: Hybrid merge (RRF)
t0 = time.time()
hybrid_results = reciprocal_rank_fusion(
    kw_results, vec_results,
    k=RRF_K, top_n=MERGE_TOP_N,
)
hybrid_ms = (time.time() - t0) * 1000

# Stage 4: Rerank
# The reranker needs Chunk objects. Look them up from LanceDB.
import lancedb
from workbench.core.types import Chunk

db = lancedb.connect(str(db_path))
table = db.open_table("chunks")
all_data = table.to_pandas()

hybrid_ids = [r.chunk_id for r in hybrid_results]
matched = all_data[all_data["chunk_id"].isin(hybrid_ids)]
chunk_map = {}
for _, row in matched.iterrows():
    chunk_map[row["chunk_id"]] = Chunk(
        chunk_id=row["chunk_id"],
        document_id=row.get("document_id", ""),
        title=row.get("title", ""),
        section=row.get("section", None),
        text=row.get("text", ""),
        start_offset=0,
        end_offset=0,
        token_count=len(row.get("text", "")) // 4,
    )

hybrid_chunks = [chunk_map[cid] for cid in hybrid_ids if cid in chunk_map]

t0 = time.time()
rerank_results = reranker.rerank(query, hybrid_chunks, top_n=RERANK_TOP_N)
rerank_ms = (time.time() - t0) * 1000

print(f"Query:    '{QUERY_TEXT}'")
print(f"Keyword:  {len(kw_results):>3} results  ({kw_ms:6.1f} ms)")
print(f"Vector:   {len(vec_results):>3} results  ({vec_ms:6.1f} ms)")
print(f"Hybrid:   {len(hybrid_results):>3} merged   ({hybrid_ms:6.1f} ms)")
print(f"Reranked: {len(rerank_results):>3} final    ({rerank_ms:6.1f} ms)")

Query:    'French Revolution'
Keyword:   20 results  (  28.0 ms)
Vector:    20 results  ( 486.3 ms)
Hybrid:    30 merged   (   0.5 ms)
Reranked:   5 final    (4625.7 ms)


## 3 — Build the comparison table

In [5]:
import pandas as pd
from IPython.display import display, HTML


def snippet(chunk_id: str, max_chars: int = SNIPPET_CHARS) -> str:
    """Get a short text snippet for a chunk_id."""
    if chunk_id in chunk_map:
        return chunk_map[chunk_id].text[:max_chars].replace("\n", " ") + "…"
    # Try to find it in all_data
    row = all_data[all_data["chunk_id"] == chunk_id]
    if len(row) > 0:
        return str(row.iloc[0]["text"])[:max_chars].replace("\n", " ") + "…"
    return ""


def title_for(chunk_id: str) -> str:
    """Get the article title for a chunk_id."""
    if chunk_id in chunk_map:
        return chunk_map[chunk_id].title
    row = all_data[all_data["chunk_id"] == chunk_id]
    if len(row) > 0:
        return str(row.iloc[0].get("title", ""))
    return ""


# Collect rank info per chunk_id across all 4 stages
# Use the VECTOR results as the row basis (k rows)
kw_rank  = {r.chunk_id: (i + 1, r.score) for i, r in enumerate(kw_results)}
vec_rank = {r.chunk_id: (i + 1, r.score) for i, r in enumerate(vec_results)}
hyb_rank = {r.chunk_id: (i + 1, r.score) for i, r in enumerate(hybrid_results)}
re_rank  = {r.chunk_id: (i + 1, r.rerank_score) for i, r in enumerate(rerank_results)}

# The rows: union of vector + keyword results so nothing is lost
seen = set()
ordered_ids = []
for r in vec_results:
    if r.chunk_id not in seen:
        ordered_ids.append(r.chunk_id)
        seen.add(r.chunk_id)
for r in kw_results:
    if r.chunk_id not in seen:
        ordered_ids.append(r.chunk_id)
        seen.add(r.chunk_id)


def fmt_cell(rank_score: tuple | None) -> str:
    """Format a (rank, score) tuple into a cell string."""
    if rank_score is None:
        return "—"
    rank, score = rank_score
    return f"#{rank}  ({score:.4f})"


rows = []
for cid in ordered_ids:
    rows.append({
        "chunk_id": cid,
        "title": title_for(cid),
        "snippet": snippet(cid),
        "Semantic": fmt_cell(vec_rank.get(cid)),
        "BM25": fmt_cell(kw_rank.get(cid)),
        "Hybrid (RRF)": fmt_cell(hyb_rank.get(cid)),
        "Reranked": fmt_cell(re_rank.get(cid)),
    })

df = pd.DataFrame(rows)
print(f"Table: {len(df)} rows  |  {RERANK_TOP_N} reranked winners (green)")

Table: 34 rows  |  5 reranked winners (green)


## 4 — Display with green highlighting

In [6]:
reranked_ids = {r.chunk_id for r in rerank_results}

HIGHLIGHT = "background-color: #c6efce; color: #006100; font-weight: bold"


def highlight_reranked(row: pd.Series) -> list[str]:
    """
    Styler function: if this row's chunk_id is in the reranked set,
    paint the ENTIRE row green.
    """
    if row["chunk_id"] in reranked_ids:
        return [HIGHLIGHT] * len(row)
    return [""] * len(row)


styled = (
    df.style
    .apply(highlight_reranked, axis=1)
    .set_properties(**{"text-align": "left", "white-space": "nowrap"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td, th", "props": [("padding", "6px 10px")]},
    ])
    .hide(axis="index")
)
styled

chunk_id,title,snippet,Semantic,BM25,Hybrid (RRF),Reranked
e319206887cdb66e,French Revolution,The French Revolution was a revolution in France from 1789 to 1799. It ended the…,#1 (0.7949),#1 (13.9861),#1 (0.0328),#1 (0.9999)
3d0459e270cc0c1c,Revolution,revolution of Pier Gerlofs Donia and Wijerd Jelckama]] A revolution is a very s…,#2 (0.7892),#19 (12.7405),#2 (0.0288),—
362cc7c5424eb284,History of Europe,"The Industrial Revolution, beginning in Great Britain, allowed masses of people,…",#3 (0.7805),—,#8 (0.0159),—
af8a7856301e0c5e,Modern history,"Inventions of the Steam engine (1764) and Spinning jenny (1769), one of the peak…",#4 (0.7754),—,#10 (0.0156),—
a41f5ef5d1b259df,French First Republic,{{Infobox former country | native_name = République française | conventional_lon…,#5 (0.7738),—,#12 (0.0154),—
2f30c2b63b87bce2,French Revolutionary Wars,The French Revolutionary Wars are conflicts between 1792 and 1802 that started b…,#6 (0.7732),—,#13 (0.0152),#4 (0.9687)
0019f2d17bb5f0b7,Human history,From nationalism to imperialism The French Revolution lead to massive political …,#7 (0.7693),—,#15 (0.0149),—
ad474cba768fc3de,July Revolution,"The French Revolution of 1830, also known as the July Revolution, was the throwi…",#8 (0.7686),#14 (12.9089),#4 (0.0282),#3 (0.9938)
ae3c503f5bf9421b,French Revolution,Causes of the revolution The problems in France that led up to the Revolution: U…,#9 (0.7672),—,#17 (0.0145),—
f400336a33f263a2,July Monarchy,The July Monarchy () was a liberal constitutional monarchy in France ruled by . …,#10 (0.7671),#13 (12.9089),#5 (0.0280),—


## 5 — Drill-down: inspect individual reranked chunks

In [7]:
print(f"=== Top {RERANK_TOP_N} reranked chunks for: '{QUERY_TEXT}' ===")
print()
for i, rr in enumerate(rerank_results, 1):
    ch = chunk_map.get(rr.chunk_id)
    title = ch.title if ch else "?"
    text_preview = ch.text[:200] if ch else "?"

    kw_info  = kw_rank.get(rr.chunk_id)
    vec_info = vec_rank.get(rr.chunk_id)
    hyb_info = hyb_rank.get(rr.chunk_id)

    print(f"── {i}. {title} ── (chunk {rr.chunk_id})")
    print(f"   Rerank score : {rr.rerank_score:.4f}")
    print(f"   Semantic rank: {vec_info[0] if vec_info else '—'}")
    print(f"   BM25 rank    : {kw_info[0] if kw_info else '—'}")
    print(f"   Hybrid rank  : {hyb_info[0] if hyb_info else '—'}")
    print(f"   Text: {text_preview}…")
    print()

=== Top 5 reranked chunks for: 'French Revolution' ===

── 1. French Revolution ── (chunk e319206887cdb66e)
   Rerank score : 0.9999
   Semantic rank: 1
   BM25 rank    : 1
   Hybrid rank  : 1
   Text: The French Revolution was a revolution in France from 1789 to 1799. It ended the French monarchy. The revolution began with a meeting of the Estates General in Versailles, and ended when Napoleon Bona…

── 2. French Revolutionary Wars ── (chunk 2fd3985bd2f7dba5)
   Rerank score : 0.9970
   Semantic rank: 15
   BM25 rank    : 5
   Hybrid rank  : 3
   Text: The French Revolution removed the King of France, King Louis XVI, from power. As early as 1791, the other monarchies of Europe hated the French Revolution and wondered if they should intervene to stop…

── 3. July Revolution ── (chunk ad474cba768fc3de)
   Rerank score : 0.9938
   Semantic rank: 8
   BM25 rank    : 14
   Hybrid rank  : 4
   Text: The French Revolution of 1830, also known as the July Revolution, was the throwing off of Ch

## 6 — Experiment: sweep RRF-k and see how ordering changes

In [8]:
print(f"Query: '{QUERY_TEXT}'")
print(f"Sweeping RRF k over [1, 10, 30, 60, 120] with top_n={MERGE_TOP_N}")
print()

for k_val in [1, 10, 30, 60, 120]:
    merged = reciprocal_rank_fusion(
        kw_results, vec_results, k=k_val, top_n=5,
    )
    ids = [r.chunk_id for r in merged]
    titles = [title_for(cid)[:30] for cid in ids]
    print(f"  k={k_val:>3}  →  {titles}")

Query: 'French Revolution'
Sweeping RRF k over [1, 10, 30, 60, 120] with top_n=30

  k=  1  →  ['French Revolution', 'Revolution', 'Jacques-Louis David', 'History of Europe', "Danton's Death"]
  k= 10  →  ['French Revolution', 'Revolution', 'French Revolutionary Wars', 'July Revolution', 'July Monarchy']
  k= 30  →  ['French Revolution', 'Revolution', 'French Revolutionary Wars', 'July Revolution', 'July Monarchy']
  k= 60  →  ['French Revolution', 'Revolution', 'French Revolutionary Wars', 'July Revolution', 'July Monarchy']
  k=120  →  ['French Revolution', 'French Revolutionary Wars', 'Revolution', 'July Revolution', 'July Monarchy']


## 7 — Experiment: keyword weight via asymmetric top-k

Instead of weighting scores (which are on different scales), we control the *balance* between keyword and semantic by changing how many candidates each contributes. More candidates = more influence after RRF.

In [9]:
configs = [
    {"label": "semantic-heavy",  "kw_k": 5,  "vec_k": 30},
    {"label": "balanced",        "kw_k": 20, "vec_k": 20},
    {"label": "keyword-heavy",   "kw_k": 30, "vec_k": 5},
]

for cfg in configs:
    kw_res  = kw_searcher.search(query, top_k=cfg["kw_k"])
    vec_res = vec_searcher.search(query, top_k=cfg["vec_k"])
    merged  = reciprocal_rank_fusion(kw_res, vec_res, k=RRF_K, top_n=5)
    titles  = [title_for(r.chunk_id)[:35] for r in merged]
    print(f"  {cfg['label']:<16}  (kw={cfg['kw_k']:>2}, vec={cfg['vec_k']:>2})  →  {titles}")

  semantic-heavy    (kw= 5, vec=30)  →  ['French Revolution', 'French Revolutionary Wars', 'Revolution', 'Jacques-Louis David', 'History of Europe']
  balanced          (kw=20, vec=20)  →  ['French Revolution', 'Revolution', 'French Revolutionary Wars', 'July Revolution', 'July Monarchy']
  keyword-heavy     (kw=30, vec= 5)  →  ['French Revolution', 'Revolution', 'Jacques-Louis David', 'History of Europe', "Danton's Death"]


## 8 — Experiment: compare reranker behaviour with different queries

In [10]:
test_queries = [
    "What caused the French Revolution?",
    "How does photosynthesis work in plants?",
    "History of the Roman Empire",
    "Python programming language features",
    "Climate change effects on ocean currents",
]

for q_text in test_queries:
    q = Query(text=q_text)

    kw_res  = kw_searcher.search(q, top_k=KEYWORD_TOP_K)
    vec_res = vec_searcher.search(q, top_k=VECTOR_TOP_K)
    merged  = reciprocal_rank_fusion(kw_res, vec_res, k=RRF_K, top_n=MERGE_TOP_N)

    # Look up chunks for reranking
    m_ids = [r.chunk_id for r in merged]
    m_chunks = [chunk_map[cid] for cid in m_ids if cid in chunk_map]

    rr = reranker.rerank(q, m_chunks, top_n=3)

    print(f"\n  Q: '{q_text}'")
    for i, r in enumerate(rr, 1):
        t = title_for(r.chunk_id)[:40]
        print(f"     {i}. {t:<40}  score={r.rerank_score:.4f}")


  Q: 'What caused the French Revolution?'
     1. French Revolution                         score=0.9591
     2. France                                    score=0.8342
     3. French Revolution                         score=0.7902

  Q: 'How does photosynthesis work in plants?'

  Q: 'History of the Roman Empire'

  Q: 'Python programming language features'

  Q: 'Climate change effects on ocean currents'


---

### Things to try next

- Change `QUERY_TEXT` at the top and re-run all cells
- Increase `KEYWORD_TOP_K` or `VECTOR_TOP_K` to 50 and see if reranked results change
- Try `RRF_K = 1` vs `RRF_K = 120` — small k amplifies top-rank advantage
- Swap reranker model: try `"cross-encoder/ms-marco-MiniLM-L-6-v2"` (faster, less accurate)
- Look at cases where BM25 finds something semantic misses (and vice versa) — that's the whole point of hybrid search